# installing libraries and preparing data 

In [30]:
import json, math, csv
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import numpy as np

In [31]:
exp_gdf = gpd.read_file("/Users/saraalsubaie/Desktop/riyadh_experiences/data/riyadh_experiences_geocoded-2.geojson")
metro_gdf = gpd.read_file("/Users/saraalsubaie/Desktop/riyadh_experiences/data/metro-stations-in-riyadh-by-metro-line-and-station-type-2024.geojson")
bus_gdf = gpd.read_file("/Users/saraalsubaie/Desktop/riyadh_experiences/data/bus-stops-in-riyadh-by-bus-route-direction-and-shelter-type-2024.geojson")
 
#et CRS to WGS84, then project to UTM Zone 38N for distance ops
CRS_GEO = "EPSG:4326"
CRS_METRIC = "EPSG:32638"   
 
exp_gdf = exp_gdf.set_crs(CRS_GEO).to_crs(CRS_METRIC)
metro_gdf = metro_gdf.set_crs(CRS_GEO).to_crs(CRS_METRIC)
bus_gdf = bus_gdf.set_crs(CRS_GEO).to_crs(CRS_METRIC)

# spatial analysis 

## Buffer analysis: how many experiences are within 500m / 1km of a metro?

In [32]:
#  buffer analysis 
from shapely.ops import unary_union

metro_buffer_500  = unary_union(metro_gdf.buffer(500))
metro_buffer_1000 = unary_union(metro_gdf.buffer(1000))
bus_buffer_300    = unary_union(bus_gdf.buffer(300))

exp_gdf["metro_500m"]  = exp_gdf.geometry.within(metro_buffer_500)
exp_gdf["metro_1km"]   = exp_gdf.geometry.within(metro_buffer_1000)
exp_gdf["bus_300m"]    = exp_gdf.geometry.within(bus_buffer_300)

print("buffer analysis: ")
print(f"Within 500m of metro:  {exp_gdf['metro_500m'].sum()} / {len(exp_gdf)}")
print(f"Within 1km of metro:   {exp_gdf['metro_1km'].sum()} / {len(exp_gdf)}")
print(f"Within 300m of bus:    {exp_gdf['bus_300m'].sum()} / {len(exp_gdf)}")
print(f"No transit access (<1km metro & >300m bus): "
      f"{(~exp_gdf['metro_1km'] & ~exp_gdf['bus_300m']).sum()}")

buffer analysis: 
Within 500m of metro:  10 / 38
Within 1km of metro:   13 / 38
Within 300m of bus:    14 / 38
No transit access (<1km metro & >300m bus): 21


## nearest metro station > sjoin

In [33]:
exp_nearest_metro = gpd.sjoin_nearest(
    exp_gdf[["name","category","geometry"]],
    metro_gdf[["metro_station_desc_en","metro_line_desc_en","geometry"]],
    distance_col="dist_to_metro_m"
)
exp_nearest_bus = gpd.sjoin_nearest(
    exp_gdf[["name","geometry"]],
    bus_gdf[["busstopname","busroute","bsheltertype","geometry"]],
    distance_col="dist_to_bus_m"
)

In [34]:
result = exp_gdf.merge(
    exp_nearest_metro[["name","metro_station_desc_en","metro_line_desc_en","dist_to_metro_m"]],
    on="name"
).merge(
    exp_nearest_bus[["name","busstopname","busroute","dist_to_bus_m"]],
    on="name"
)

## cluster analysis - DBSCAN

In [35]:
from sklearn.cluster import DBSCAN
coords = np.array([(g.x, g.y) for g in exp_gdf[exp_gdf["name"] != "King Khaled Royal Reserve (Urmah)"].geometry])
db = DBSCAN(eps=2000, min_samples=3).fit(coords)    # 2km radius clusters
exp_gdf.loc[exp_gdf["name"] != "King Khaled Royal Reserve (Urmah)", "cluster"] = db.labels_
print("DBSCAN clusters (eps=2km, min_samples=3):")
print(exp_gdf.groupby("cluster")["name"].apply(list))

DBSCAN clusters (eps=2km, min_samples=3):
cluster
-1.0    [Riyadh National Zoo, Via Riyadh, Wadi Namar W...
 0.0    [Murabba Palace, Addoho Neighborhood, Al Masma...
 1.0    [At-Turaif World Heritage Site, Diriyah Histor...
Name: name, dtype: object


In [36]:
#whihc metro lines serve the most experiences (nearest station)?
line_counts = result.groupby("metro_line_desc_en")["name"].count().sort_values(ascending=False)
print("EXPERIENCES SERVED BY METRO LINE (nearest)")
print(line_counts)

EXPERIENCES SERVED BY METRO LINE (nearest)
metro_line_desc_en
Blue line      12
Red line       12
Orange line     8
Yellow line     6
Purple line     1
Name: name, dtype: int64


# non spatial analysis 

In [37]:
# category breakdown
print(exp_gdf["category"].value_counts())

category
Heritage & History        6
Art & Culture             6
Museums                   5
Nature & Parks            4
Shopping & Retail         3
Dining & Entertainment    3
Urban Landmark            3
Sports & Recreation       3
Religious Sites           3
Education                 2
Name: count, dtype: int64


In [39]:
# Merge nearest metro / bus attributes onto exp_gdf so the folium popups & groupby work
if "nearest_metro" not in exp_gdf.columns:
    exp_gdf = exp_gdf.merge(
        exp_nearest_metro[["name", "metro_station_desc_en", "metro_line_desc_en", "dist_to_metro_m"]]
            .rename(columns={"metro_station_desc_en": "nearest_metro",
                             "metro_line_desc_en":    "metro_line"}),
        on="name", how="left",
    )
if "nearest_bus" not in exp_gdf.columns:
    exp_gdf = exp_gdf.merge(
        exp_nearest_bus[["name", "busstopname", "dist_to_bus_m"]]
            .rename(columns={"busstopname": "nearest_bus"}),
        on="name", how="left",
    )
exp_gdf["metro_dist_km"] = (exp_gdf["dist_to_metro_m"] / 1000).round(2)
exp_gdf["bus_dist_km"]   = (exp_gdf["dist_to_bus_m"]   / 1000).round(2)

# quick sanity check — should see real station names now
print(exp_gdf[["name", "nearest_metro", "metro_line", "metro_dist_km", "nearest_bus", "bus_dist_km"]].head())

In [40]:
cat_transit = exp_gdf.groupby("category").agg(
    count=("name","count"),
    metro_accessible=("metro_1km","sum"),
    bus_accessible=("bus_300m","sum"),
    avg_metro_dist=("metro_dist_km","mean")
).round(2)
print("transit access by category:")
print(cat_transit.sort_values("avg_metro_dist"))

transit access by category:
                        count  metro_accessible  bus_accessible  \
category                                                          
Education                   2                 1               0   
Urban Landmark              3                 2               1   
Religious Sites             3                 2               1   
Shopping & Retail           3                 0               1   
Museums                     5                 1               1   
Art & Culture               6                 1               3   
Dining & Entertainment      4                 0               0   
Sports & Recreation         3                 1               1   
Nature & Parks              4                 1               2   
Heritage & History          6                 4               4   

                        avg_metro_dist  
category                                
Education                         0.76  
Urban Landmark                    0.92  
Rel

In [41]:
#underserved exp
underserved = result[
    (result["dist_to_metro_m"] > 2000) & (result["dist_to_bus_m"] > 500)
][["name","category","metro_line_desc_en","dist_to_metro_m","dist_to_bus_m"]]
underserved["dist_to_metro_km"] = (underserved["dist_to_metro_m"] / 1000).round(2)
print("underserved experiences (>2km metro, >500m bus)")
print(underserved[["name","category","dist_to_metro_km"]].to_string(index=False))

underserved experiences (>2km metro, >500m bus)
                             name               category  dist_to_metro_km
             Wadi Namar Waterfall         Nature & Parks              4.64
    At-Turaif World Heritage Site     Heritage & History              6.07
           Diriyah History Museum                Museums              6.03
                  Bujairi Terrace Dining & Entertainment              6.25
King Khaled Royal Reserve (Urmah)         Nature & Parks             31.74
                   Boulevard City Dining & Entertainment              3.87
                   Boulevard City Dining & Entertainment              3.87
  King Abdulaziz Equestrian Field    Sports & Recreation              8.85
                   KAPSARC Mosque        Religious Sites              2.40
                       Dirab Park    Sports & Recreation             13.73
                     Al Samhaniya          Art & Culture              6.77
           Museum of Bygone Years                Mus

# visualizations 

## folium 

In [43]:
import folium
from folium.plugins import MarkerCluster
 
# WGS84 for web mapping
exp_wgs = exp_gdf.to_crs(CRS_GEO)
metro_wgs = metro_gdf.to_crs(CRS_GEO)
bus_wgs = bus_gdf.to_crs(CRS_GEO)
 
m = folium.Map(location=[24.7136, 46.6753], zoom_start=12, tiles="CartoDB positron")
 
LINE_COLORS = {
    "Blue line":   "#1F77B4",
    "Red line":    "#D62728",
    "Green line":  "#2CA02C",
    "Yellow line": "#BCBD22",
    "Orange line": "#FF7F0E",
    "Purple line": "#9467BD",
}
CAT_COLORS = {
    "Heritage & History": "#8B4513",
    "Museums":            "#6A0DAD",
    "Art & Culture":      "#FF69B4",
    "Nature & Parks":     "#228B22",
    "Dining & Entertainment": "#FF8C00",
    "Shopping & Retail":  "#1E90FF",
    "Sports & Recreation":"#DC143C",
    "Urban Landmark":     "#2F4F4F",
    "Religious Sites":    "#DAA520",
    "Education":          "#00CED1",
}
 
#metro stations layer
metro_group = folium.FeatureGroup(name="Metro Stations", show=True)
for _, row in metro_wgs.iterrows():
    color = LINE_COLORS.get(row["metro_line_desc_en"], "#555")
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5, color=color, fill=True, fill_opacity=0.85,
        popup=f"<b>{row['metro_station_desc_en']}</b><br>{row['metro_line_desc_en']}<br>Type: {row['metro_station_type_desc_en']}"
    ).add_to(metro_group)
metro_group.add_to(m)
 
#bus stops (clustered — 3k points)
bus_group = folium.FeatureGroup(name="Bus Stops", show=False)
mc = MarkerCluster()
for _, row in bus_wgs.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3, color="#888", fill=True, fill_opacity=0.5,
        popup=f"{row['busstopname']}<br>Route {row['busroute']}<br>{row['bsheltertype']}"
    ).add_to(mc)
mc.add_to(bus_group)
bus_group.add_to(m)
 
#experiences layer
exp_group = folium.FeatureGroup(name="Riyadh Experiences", show=True)
for _, row in exp_wgs.iterrows():
    if pd.isna(row.geometry.y): continue
    color = CAT_COLORS.get(row["category"], "#333")
    metro_d = row.get("metro_dist_km", "?")
    bus_d   = row.get("bus_dist_km", "?")
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=folium.Popup(
            f"<b>{row['name']}</b><br><i>{row['category']}</i><br><br>"
            f"Nearest metro: {row.get('nearest_metro','?')} ({metro_d} km)<br>"
            f"Metro line: {row.get('metro_line','?')}<br>"
            f"Nearest bus: {row.get('nearest_bus','?')} ({bus_d} km)<br><br>"
            f"<small>{row['description'][:120]}...</small>",
            max_width=280
        ),
        icon=folium.Icon(color="white", icon_color=color, icon="info-sign", prefix="glyphicon")
    ).add_to(exp_group)
exp_group.add_to(m)
 
#1km metro buffers for experiences
buffer_group = folium.FeatureGroup(name="1km Metro Buffers", show=False)
for _, row in exp_wgs.iterrows():
    if pd.isna(row.geometry.y): continue
    folium.Circle(
        location=[row.geometry.y, row.geometry.x],
        radius=1000, color="#1F77B4", fill=True, fill_opacity=0.05,
        weight=1
    ).add_to(buffer_group)
buffer_group.add_to(m)
 
folium.LayerControl(collapsed=False).add_to(m)
m.save("riyadh_transit_map.html")

 

## Connect metro stations into lines (colored polylines)

In [ ]:
import numpy as np

def _order_stations(coords):
    """Chain points: pick an endpoint via PCA, then greedy nearest-neighbor."""
    n = len(coords)
    if n < 2:
        return list(range(n))
    centered = coords - coords.mean(axis=0)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    projections = centered @ vt[0]
    start = int(np.argmin(projections))
    d = np.linalg.norm(coords[:, None] - coords[None, :], axis=2)
    np.fill_diagonal(d, np.inf)
    order = [start]
    remaining = set(range(n)) - {start}
    while remaining:
        nxt = min(remaining, key=lambda i: d[order[-1], i])
        order.append(nxt)
        remaining.remove(nxt)
    return order

metro_lines_paths = {}
for line_name, group in metro_wgs.groupby("metro_line_desc_en"):
    coords = np.array([[p.x, p.y] for p in group.geometry])
    order = _order_stations(coords)
    metro_lines_paths[line_name] = [(coords[i][1], coords[i][0]) for i in order]

lines_group = folium.FeatureGroup(name="Metro Lines", show=True)
for line_name, path in metro_lines_paths.items():
    folium.PolyLine(
        locations=path,
        color=LINE_COLORS.get(line_name, "#555"),
        weight=4, opacity=0.8, tooltip=line_name,
    ).add_to(lines_group)
lines_group.add_to(m)

# Refresh layer control so the new group is toggleable
for key in list(m._children):
    if key.startswith("layer_control"):
        del m._children[key]
folium.LayerControl(collapsed=False).add_to(m)

m.save("riyadh_transit_map.html")
print(f"Built {len(metro_lines_paths)} metro lines: {list(metro_lines_paths)}")

# Gradio dashboard

In [ ]:
import gradio as gr
import plotly.express as px
import html as _html

# --- Prepare display tables ---------------------------------------------
exp_display = exp_gdf.to_crs(CRS_GEO).copy()
exp_display["longitude"] = exp_display.geometry.x.round(6)
exp_display["latitude"]  = exp_display.geometry.y.round(6)

display_df = exp_display[[
    "name", "category", "nearest_metro", "metro_line", "metro_dist_km",
    "nearest_bus", "bus_dist_km", "metro_1km", "bus_300m",
    "longitude", "latitude",
]].rename(columns={
    "name": "Experience", "category": "Category",
    "nearest_metro": "Nearest Metro", "metro_line": "Metro Line",
    "metro_dist_km": "Metro Dist (km)",
    "nearest_bus": "Nearest Bus", "bus_dist_km": "Bus Dist (km)",
    "metro_1km": "Metro <1km", "bus_300m": "Bus <300m",
})

n_total     = len(exp_gdf)
n_metro_1km = int(exp_gdf["metro_1km"].sum())
n_bus_300m  = int(exp_gdf["bus_300m"].sum())
n_under     = int(((~exp_gdf["metro_1km"]) & (~exp_gdf["bus_300m"])).sum())

# --- Charts -------------------------------------------------------------
cat_counts = (exp_gdf["category"].value_counts()
              .rename_axis("Category").reset_index(name="Count"))
fig_cat = px.bar(cat_counts, x="Count", y="Category", orientation="h",
                 title="Experiences by category", color="Category",
                 color_discrete_map=CAT_COLORS)
fig_cat.update_layout(showlegend=False, yaxis={"categoryorder": "total ascending"},
                     height=380, margin=dict(l=10, r=10, t=40, b=10))

fig_transit = px.bar(
    cat_transit.reset_index(),
    x="category", y="avg_metro_dist",
    title="Average distance to nearest metro by category (km)",
    color="category", color_discrete_map=CAT_COLORS,
)
fig_transit.update_layout(showlegend=False, xaxis_tickangle=-35,
                         height=380, margin=dict(l=10, r=10, t=40, b=80))

underserved_display = underserved.rename(columns={
    "name": "Experience", "category": "Category",
    "metro_line_desc_en": "Metro Line",
    "dist_to_metro_km": "Metro Dist (km)",
})[["Experience", "Category", "Metro Line", "Metro Dist (km)"]]

# --- Embed folium map via iframe srcdoc ---------------------------------
# gr.HTML can't render nested <html>; sandboxing in an iframe fixes that.
with open("riyadh_transit_map.html") as fh:
    map_html = fh.read()
iframe_html = (
    f'<iframe srcdoc="{_html.escape(map_html, quote=True)}" '
    f'style="width:100%; height:620px; border:none; border-radius:8px;"></iframe>'
)

# --- UI -----------------------------------------------------------------
with gr.Blocks(title="Riyadh Experiences · Transit Accessibility",
               theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Riyadh Experiences · Transit Accessibility")
    gr.Markdown("Spatial analysis of 38 Riyadh experiences vs. metro & bus networks.")

    with gr.Row():
        gr.Markdown(f"### {n_total}\nExperiences")
        gr.Markdown(f"### {n_metro_1km}\nWithin 1 km of metro")
        gr.Markdown(f"### {n_bus_300m}\nWithin 300 m of bus")
        gr.Markdown(f"### {n_under}\nUnderserved")

    with gr.Tab("Map"):
        gr.HTML(iframe_html)

    with gr.Tab("Categories"):
        with gr.Row():
            gr.Plot(fig_cat)
            gr.Plot(fig_transit)

    with gr.Tab("Transit by category"):
        gr.Dataframe(cat_transit.reset_index(), interactive=False, wrap=True)

    with gr.Tab("Underserved"):
        gr.Markdown("More than 2 km from a metro **and** more than 500 m from a bus stop.")
        gr.Dataframe(underserved_display, interactive=False, wrap=True)

    with gr.Tab("All experiences"):
        gr.Dataframe(display_df, interactive=False, wrap=True)

demo.launch(inline=True, share=False)